In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [20]:
from pathlib import Path

data = Path('/kaggle/input/playground-series-s6e1/')

df_train = pd.read_csv(data / 'train.csv')
df_test = pd.read_csv(data / 'test.csv')

In [21]:
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, StandardScaler

scaler = StandardScaler()
onehot = OneHotEncoder(sparse_output=False)
target = 'exam_score'

def preprocess(X, scaler, onehot, fit=True):
    if target in X.columns.tolist():
        X = X.drop(columns=[target, 'id'])
    else:
        X = X.drop(columns=['id'])

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    
    if fit:
        scaled = scaler.fit_transform(X[num_cols])
    else:
        scaled = scaler.transform(X[num_cols])
        
    X = X.drop(columns=num_cols)
    X[num_cols] = scaled
    
    cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

    if fit:
        encoded = onehot.fit_transform(X[cat_cols])
    else:
        encoded = onehot.transform(X[cat_cols])
    
    new_cols = onehot.get_feature_names_out()

    X = X.drop(columns=cat_cols)
    X[new_cols] = encoded

    return X

In [22]:
X_train_full = preprocess(df_train[:600_000], scaler, onehot, fit=True)
y_train_full = df_train[:600_000][target]
X_train = preprocess(df_train[:60_000], scaler, onehot, fit=True)
y_train = df_train[:60_000][target]
X_valid = preprocess(df_train[600_000:], scaler, onehot, fit=False)
y_valid = df_train[600_000:][target]

In [23]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics

torch.manual_seed(42)

X_train_full_ts = torch.tensor(X_train_full.values, dtype=torch.float32)
y_train_full_ts = torch.tensor(y_train_full.values, dtype=torch.float32).unsqueeze(1)

ds_train_full = TensorDataset(X_train_full_ts, y_train_full_ts)
train_full_loader = DataLoader(
    ds_train_full,
    batch_size=32,
    shuffle=True,
    pin_memory=True
)

X_train_ts = torch.tensor(X_train.values, dtype=torch.float32)
y_train_ts = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

ds_train = TensorDataset(X_train_ts, y_train_ts)
train_loader = DataLoader(
    ds_train,
    batch_size=32,
    shuffle=True,
    pin_memory=True
)

X_valid_ts = torch.tensor(X_valid.values, dtype=torch.float32)
y_valud_ts = torch.tensor(y_valid.values, dtype=torch.float32).unsqueeze(1)

ds_valid = TensorDataset(X_valid_ts, y_valud_ts)
valid_loader = DataLoader(
    ds_valid,
    batch_size=32,
    pin_memory=True
)

In [24]:
class TestPredictMLP(nn.Module):
    def __init__(self, n_in, n_h1, n_h2, n_out):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_in, n_h1),
            nn.ReLU(),
            # nn.Dropout(dropout_rate),
            nn.Linear(n_h1, n_h2),
            nn.ReLU(),
            # nn.Dropout(dropout_rate),
            nn.Linear(n_h2, n_out)
        )

    def forward(self, X):
        return self.mlp(X)

In [25]:
def train(model, dataloader, optimizer, criterion, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        mean_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {mean_loss}")

def evaluate(model, dataloader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [26]:
# import optuna

# if torch.cuda.is_available():
#     device = 'cuda'
# else:
#     device = 'cpu'
# print(device)

# mse = nn.MSELoss()

# def objective(trial):
#     lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
#     n_h = trial.suggest_int('n_h', 20, 300)
#     w_d = trial.suggest_float('w_d', 1e-6, 1e-1, log=True)
#     model = TestPredictMLP(n_in=X_train_ts.shape[1], n_h1=n_h, n_h2=n_h, n_out=1).to(device)
#     # nadam = torch.optim.NAdam(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=0, momentum_decay=0.004)
#     adamw = torch.optim.AdamW(
#         model.parameters(),
#         lr=lr,
#         weight_decay=w_d
#     )
#     train(model, train_loader, adamw, mse, 5)

#     rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
#     rmse_loss = evaluate(model, valid_loader, rmse)

#     return rmse_loss

# torch.manual_seed(42)
# sampler = optuna.samplers.TPESampler(seed=42)
# study = optuna.create_study(direction='minimize', sampler=sampler)
# study.optimize(objective, n_trials=8)

In [28]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(device)

n_in = X_train_ts.shape[1]
n_h1 = 64
n_h2 = 128
n_out = 1

model = TestPredictMLP(n_in, n_h1, n_h2, n_out).to(device)
# sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
# nadam = torch.optim.NAdam(model.parameters(), lr=0.002, betas=(0.9, 0.999), weight_decay=0, momentum_decay=0.004)
adamw = torch.optim.AdamW(
    model.parameters(),
    lr=3.613894271216525e-05,
    weight_decay=6.789053271698483e-05
)
mse = nn.MSELoss()
n_epochs = 8

train(model, train_full_loader, adamw, mse, n_epochs)

rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate(model, valid_loader, rmse)

# loss = criterion(torch.tensor([-1.]), torch.tensor([1.]))
# print(loss.item())

Epoch 1/5 | Loss: 362.54021283376056
Epoch 2/5 | Loss: 79.28705659291585
Epoch 3/5 | Loss: 79.20108163747152
Epoch 4/5 | Loss: 79.15493460428874
Epoch 5/5 | Loss: 79.11645550079346


tensor(8.8861, device='cuda:0')

In [29]:
X_test = preprocess(df_test, scaler, onehot, fit=False)
X_test_ts = torch.tensor(X_test.values, dtype=torch.float32)
ids = df_test['id'].copy()

test_loader = DataLoader(
    TensorDataset(X_test_ts),
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

model.eval()
preds_list = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)
        y_pred = model(X_batch)
        preds_list.append(y_pred.squeeze(1).detach().cpu().numpy()) #safety detach

test_preds = np.concatenate(preds_list, axis=0)

submission = pd.DataFrame({
    "id": ids,
    "exam_score": test_preds
})

submission.to_csv("submission.csv", index=False)